In [1]:
import torch
import numpy as np
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import random
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.common.exceptions import NoSuchElementException
from selenium.webdriver.support import expected_conditions as EC
from IPython.display import clear_output
from webdriver_manager.chrome import ChromeDriverManager
import time


# Inicializa navegador
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get("https://contexto.me/en/")

print("Carregando BERT...")
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()


Carregando BERT...


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [2]:
# Embeddings cache
embeddings = {}
memory = {}

In [3]:
# Pre-computa embeddings ou carrega do arquivo se ja existir
EMBEDDINGS_FILE = "embeddingsValues.npz"

if os.path.exists(EMBEDDINGS_FILE):
	print("Arquivo de embeddings encontrado.")
	data = np.load(EMBEDDINGS_FILE, allow_pickle=True)
	candidatos_lista = list(data["candidatos"])
	matriz_candidatos = data["matriz"]
	for word, vec in zip(candidatos_lista, matriz_candidatos):
		embeddings[word] = vec
	print(f"Pronto! {len(candidatos_lista)} candidatos carregados do arquivo.")
else:
	print("Arquivo nao encontrado. Pre-computando embeddings")
	todos_candidatos = [w for w in tokenizer.vocab.keys() if w.isalpha() and len(w) > 4]

	batch_size = 64
	candidatos_lista = []
	matriz_candidatos = []

	for i in range(0, len(todos_candidatos), batch_size):
		batch = todos_candidatos[i:i + batch_size]
		with torch.no_grad():
			inputs = tokenizer(batch, return_tensors="pt", truncation=True, padding=True)
			outputs = model(**inputs)
			embs = outputs.last_hidden_state.mean(dim=1).numpy()
		for j, word in enumerate(batch):
			candidatos_lista.append(word)
			matriz_candidatos.append(embs[j])
			embeddings[word] = embs[j]

	matriz_candidatos = np.array(matriz_candidatos)
	np.savez(EMBEDDINGS_FILE, candidatos=candidatos_lista, matriz=matriz_candidatos)
	print(f"Pronto! {len(candidatos_lista)} candidatos pre-computados e salvos em '{EMBEDDINGS_FILE}'.")

Arquivo de embeddings encontrado.
Pronto! 18348 candidatos carregados do arquivo.


In [4]:
# Palavras genéricas para abrir sentidos semânticos
temas_exploratorios = [
	"object", "place", "person", "animal", "emotion",
	"food", "vehicle", "technology", "family", "music",
	"clothing", "plant", "profession", "color"
]
# "tool"
temas_usados = set()
palavras_usadas = set()

# Função para obter embedding
def get_embedding(word):
	if word in embeddings:
		return embeddings[word]

	with torch.no_grad():
		inputs = tokenizer(word, return_tensors="pt", truncation=True)
		inputs = {k: v for k, v in inputs.items()}
		outputs = model(**inputs)
		emb = outputs.last_hidden_state.mean(dim=1).squeeze().numpy()
		embeddings[word] = emb
		return emb

# Adiciona chute manual
def adicionar_chute(palavra, score):
	memory[palavra] = score
	palavras_usadas.add(palavra)
	get_embedding(palavra)

# Sugestão adaptativa
def sugerir_proximo():
	if not memory:
		sugestao = random.choice(temas_exploratorios)
		temas_usados.add(sugestao)
		palavras_usadas.add(sugestao)
		return [sugestao]

	# Classificação refinada por score
	perfeitos = sorted([(p, s) for p, s in memory.items() if s < 50],         key=lambda x: x[1])
	otimos    = sorted([(p, s) for p, s in memory.items() if 50  <= s < 200], key=lambda x: x[1])
	bons      = sorted([(p, s) for p, s in memory.items() if 200 <= s < 300], key=lambda x: x[1])
	oks       = sorted([(p, s) for p, s in memory.items() if 300 <= s < 500], key=lambda x: x[1])
	ruins     = [(p, s) for p, s in memory.items() if s >= 500]

	todos_ruins = len(ruins) == len(memory)

	if todos_ruins:
		restantes = [t for t in temas_exploratorios if t not in temas_usados]
		if restantes:
			sugestao = random.choice(restantes)
			temas_usados.add(sugestao)
			palavras_usadas.add(sugestao)
			print(f"[explorando novo tema: {sugestao}]")
			return [sugestao]
		else:
			candidato = random.choice([c for c in candidatos_lista if c not in palavras_usadas])
			palavras_usadas.add(candidato)
			return [candidato]

	MIN_BASE = 10
	base = []

	if len(perfeitos) > 0:
		base = list(perfeitos)
		if len(base) < MIN_BASE:
			base.extend(otimos[:MIN_BASE - len(base)])
	elif len(otimos) > 0:
		base = list(otimos)
		if len(base) < MIN_BASE:
			base.extend(bons[:MIN_BASE - len(base)])
	elif len(bons) > 0:
		base = list(bons)
		if len(base) < MIN_BASE:
			base.extend(oks[:MIN_BASE - len(base)])
	elif len(oks) > 0:
		base = list(oks)
	else:
		base = sorted(memory.items(), key=lambda x: x[1])[:MIN_BASE]

	# Cálculo do vetor médio com pesos
	vetores = np.array([get_embedding(p) for p, _ in base])
	pesos = np.array([1 / (s + 1) for _, s in base])
	pesos = pesos / pesos.sum()
	vetor_medio = np.average(vetores, axis=0, weights=pesos)

	# Seleciona candidatos nao testados usando a matriz pre-computada
	mask = np.array([c not in memory and c not in palavras_usadas for c in candidatos_lista])
	candidatos_filtrados = [candidatos_lista[i] for i in range(len(candidatos_lista)) if mask[i]]
	matriz_filtrada = matriz_candidatos[mask]

	# Similaridade em batch — sem loop, sem BERT
	sims = cosine_similarity([vetor_medio], matriz_filtrada)[0]
	top10_indices = np.argsort(sims)[-10:][::-1]
	lista_palavras = [candidatos_filtrados[i] for i in top10_indices]
	palavras_usadas.update(lista_palavras)
	return lista_palavras

def enviar_chute_site(palavra):
	palavra = str(palavra)
	input_box = driver.find_element(By.CSS_SELECTOR, "input.word")
	input_box.clear()
	input_box.send_keys(palavra)
	input_box.send_keys(Keys.ENTER)

	# Verifica se apareceu a mensagem de palavra repetida
	try:
		msg_box = driver.find_element(By.CSS_SELECTOR, ".message-text")
		if "already guessed" in msg_box.text:
			print('Repetida:', palavra)
			palavras_usadas.add(palavra)
			return palavra, None
	except:
		pass  # nenhuma mensagem = segue normal

	# Aguarda o resultado aparecer no DOM (timeout de 5s)
	try:
		WebDriverWait(driver, 5).until(
			EC.presence_of_element_located((By.CSS_SELECTOR, ".row-wrapper.current .row"))
		)
		resultado = driver.find_element(By.CSS_SELECTOR, ".row-wrapper.current .row")
		texto = resultado.text.strip()
		palavra_retornada, score = texto.split()
		return palavra_retornada.lower(), int(score)
	except Exception as e:
		print(f"[erro ao ler resultado de '{palavra}': {e}]")
		return palavra, None

def reenviar_chutes_para_interface():
	print("Recarregando chutes anteriores...")
	for palavra in list(memory.keys()):
		enviar_chute_site(palavra)

def adicionar_chute_site(palavra):
	chute, score = enviar_chute_site(palavra)

	if score is None:
		return

	memory[chute] = score
	palavras_usadas.add(chute)
	get_embedding(chute)
	print(f"{chute} → {score}")

In [5]:
time.sleep(5)
def loop_automatico():
	reenviar_chutes_para_interface()
	while True:
		proximas = sugerir_proximo()
		for palavra in proximas:
			chute, score = enviar_chute_site(palavra)

			if score is not None:
				adicionar_chute(chute, score)
				print(f"{chute} → {score}")
				if score == None:
					print('Repetida:', palavra)
				if score <= 5:
					print(f"[top 5!]")
				if score == 1:
					print(f"Palavra encontrada: {chute}")
					return


In [6]:
listaPalavras = []

if len(listaPalavras) == 0:
    print("Nenhuma palavra foi passada para o loop automático.") 
    for palavra in listaPalavras:
        adicionar_chute_site(palavra)
        print(f"Enviando chute manual: {palavra}")
else:
    print("Iniciando jogo")
try:
	loop_automatico()

except NoSuchElementException:
	clear_output(wait=False)
	print("\n[ERRO] Não foi possível encontrar o campo de input do jogo.")
	print("Provavelmente você ainda NÃO está na tela do jogo.")
	print("\n→ Entre na partida manualmente no navegador (onde aparece o campo de digitar palavra)")
	print("→ Depois execute o script novamente.\n")

Nenhuma palavra foi passada para o loop automático.
Recarregando chutes anteriores...
emotion → 1542
[explorando novo tema: vehicle]
emotion → 1542
[explorando novo tema: plant]
vehicle → 5819
[explorando novo tema: technology]
plant → 3031
[explorando novo tema: person]
technology → 1980
[explorando novo tema: food]
person → 636
[explorando novo tema: profession]
food → 867
[explorando novo tema: place]
profession → 1778
[explorando novo tema: family]
place → 374
family → 839
place → 374
position → 1612
hometown → 9711
indication → 8786
pinnacle → 4813
inform → 3495
stint → 6495
chef → 2817
accomplishment → 14459
round → 3032
place → 374
highlight → 2512
order → 2002
competitor → 5781
maroon → 9790
classify → 21589
group → 554
footprint → 7611
suggestion → 797
select → 2990
office → 3482
failing → 5676
persistent → 10674
big → 2202
rockies → 9731
rockies → 9731
rich → 6019
suspend → 11130
proficient → 13392
regatta → 10417
zealand → 1894
subcontinent → 12051
highlight → 2512
affect → 

AttributeError: 'NoneType' object has no attribute 'clear'